In [ ]:
import pandas as pd 
import numpy as np

DOY = 218 
shortwave = 0 #800 # Shortwave radiation in W/m^2
T_air_c   = 17  # air temperature in Celsius
T_water_C = 28  # Surface temperature in Celsius
RH = 74
wind_speed = 1.183 

 
month = 8



altitude = 4        # Altitude of Stockton in meters above sea level 
latitude = 37.5     # Latitude of Stockton in degrees 

# CONSTANTS 
# [1] for shortwave
Tropic = 23.45 
days_per_year = 365.0       
DclDay = 284.0
DgCrcl = 360.0 
deg2rad = 57.29577951 
RefInd = 1.33                        # This is a mild function of T and S. http://scubageek.com/articles/wwwh2o.html
latitude_radian = latitude/deg2rad

# [2] for longwave
surface_pressure = 101325 * (1 - altitude * 2.25577e-5)**5.25588 * (1/100) # calculated in (mb)
G = 2.92                            # Values from Smith-Gamma table (need to look up)# Other option for springtime is 3.11 
epsilon_w = 0.972                   # Emissivity of water (Davies et al., 1971)
sigma    = 5.67e-8                  # Stefan-Boltzmann constant (W m^-2 K^-4)


# [3] for heat flux
C_pa = 1006 # Specific heat of air at constant pressure (J kg^-1 C^-1)
# Lv = 2.501e6 - 2370*T_water_C      # Latent heat of vaporization (J kg^-1)
Lv = (2.5-0.00234*T_water_C)*1.e6 # Latent heat of vaporization (J kg^-1) [GOTM]

In [70]:
def calculate_dew_point(vapor_pressure):
    """Calculate the dew point temperature from vapor pressure."""
    Td1 = 243.5 * np.log(vapor_pressure / 6.112)
    Td2 = 17.67 - np.log(vapor_pressure / 6.112)
    T_dew_point = Td1 / Td2 # Degrees Celsius 
    T_dew_point_F = T_dew_point*9/5+32 # Convert to Fahrenheit
    return T_dew_point_F

def celsius2kelvin(celsius):
    """Convert Celsius to Kelvin."""
    return celsius + 273.13


def calculate_zenith(DOY):
    z = DclDay + np.floor(DOY)
    x = DgCrcl * z/days_per_year/deg2rad
    y = np.sin(x)
    Decl = Tropic/deg2rad * y

    # Hour-angle calculation where time is hours from midnight  
    HrAng = ((DOY-np.floor(DOY))*24 -12)*15.0/deg2rad

    # Zenith angle calculation 
    zenith = np.acos(np.sin(Decl)*np.sin(latitude_radian)+np.cos(Decl)*np.cos(latitude_radian)*np.cos(HrAng))
    return zenith 

def calculate_albedo(zenith):
    # Angle of Refraction calculation based on Snell's Law 
    RefAng = np.asin(np.sin(zenith)/RefInd)  # Angle of refraction

    # Albedo Calculation 
    A1 = np.tan(zenith - RefAng)**2
    A2 = np.tan(zenith + RefAng)**2
    A3 = np.sin(zenith - RefAng)**2
    A4 = np.sin(zenith + RefAng)**2
    albedo = 0.5 * (A1/A2 + A3/A4)

    albedo = np.clip(albedo, 0, 1)  # Ensure albedo is between 0 and 1
    return albedo 

def get_saturation_vapor_pressure(T_air_c):
    # Empirical fit from P.R. Lowe, 1976. An Approximating Polynomial for the
    # Computation of Saturation Vapor Pressure. Journal of Applied Meteorology
    a0 = 6.107799961
    a1 = 4.436518521e-1
    a2 = 1.428945805e-2
    a3 = 2.650648471e-4
    a4 = 3.031240396e-6
    a5 = 2.034080948e-8
    a6 = 6.136820929e-11

    # Saturation vapor pressure of the air 
    sat_pressure = a0 + T_air_c *(a1 + T_air_c*(a2 + T_air_c*(a3 + T_air_c*(a4 + T_air_c*(a5 + a6*T_air_c)))))
    return sat_pressure


In [ ]:
T_air_k =  celsius2kelvin(T_air_c)  # air temperature in Kelvin
T_water_k  = celsius2kelvin(T_water_C)  # Convert surface temperature to Kelvin
sat_pressure = get_saturation_vapor_pressure(T_air_c)   # Saturation vapor pressure in hPa
vapor_pressure = RH/100 * sat_pressure                  # Vapor pressure 
T_dew_point_F = calculate_dew_point(vapor_pressure)  # Dew point temperature in Fahrenheit
print("The dew point temperature is: #2.2f F" % T_dew_point_F)

# Shortwave radiation in (W/m^2)
zenith = calculate_zenith(DOY)
albedo = calculate_albedo(zenith)
Qsin = shortwave * (1 - albedo)

# Longwave radiation out (W/m^2)
Qlout = epsilon_w * sigma * T_water_k**4    
print("Longwave radiation out is: %2.2f W/m^2" % Qlout)

# Calculate longwave radiation in (W/m^2)
# Incoming long-wave radiation is estimated following the methods of Crawford and Duchon (1999) 
# as used in other limnological studies (e.g. Jakkila et al., 2009 Read et al., 2012)

cos_zenith = np.cos(zenith)                             # Cosine of the solar zenith angle
print("Cosine of zenith angle is: %2.2f" % cos_zenith)
air_mass_thickness_coeff = 35 * (1244 * cos_zenith**2 + 1)**(-1/2) 
precip_water = np.exp((0.1133 - np.log(G+1)) + 0.0393*T_dew_point_F) # Note T_dew_point is in Fahrenheit 

# Aerosol attenuation calculated from clear-sky conditions 
T_a = 0.95**air_mass_thickness_coeff   # Houghton (1954)

Tr_x_Tpg = 1.021 - 0.084*np.sqrt(air_mass_thickness_coeff*(0.000949*surface_pressure + 0.0151))

# Water vapor absorption in the atmosphere from McDonald (1960) 
T_w = 1 - 0.077*(precip_water * air_mass_thickness_coeff)**(0.3)


I_eff = 1353 * (1 + 0.034 * np.cos(2*np.pi * (DOY-1)/365))**2  # (Meyers and Dale, 1983) # Sw note cos might be^2 ? 
print("I_eff = %2.2f" % I_eff)
clear_sky_shortwave =  I_eff * cos_zenith * Tr_x_Tpg * T_w * T_a # Clear-sky shortwave radiation (W m^-2)

# print("T_w = %2.2f" % T_w)
# print("Tr_x_Tpg = %2.2f" % Tr_x_Tpg)
print("clear_sky_shortwave = %2.2f" % clear_sky_shortwave)

s = shortwave/clear_sky_shortwave  # Ratio of measured shortwave radiation to the clear-sky shortwave radiation
clf = 1 - s     # Cloud cover fraction 

term1 = sigma * celsius2kelvin(T_air_c)**4 
term2 = 1.22 + 0.06 * np.sin((month + 2)* np.pi/6)
term3 = (vapor_pressure/celsius2kelvin(T_air_c))**(1/7)

Qlin = term1 * (clf + (1-clf) * term2 * term3) 
print("Qlin = %2.2f W/m^2" % Qlin)



The dew point temperature is: 54.21 F
Longwave radiation out is: 453.17 W/m^2
Cosine of zenith angle is: -0.59
I_eff = 1277.82
clear_sky_shortwave = -554.62
Qlin = 401.75 W/m^2


In [ ]:
# GOTM FORMULATION FROM KONDO

SpecificHeatAir = 1006 # Specific heat capacity of air, J kg-1 K-1
gas_constant = 287.1   # gas constant for dry air J kg-1 K-1
const06=0.62198

Lv = (2.5-0.00234*T_water_C)*1.e6

e_sat = 6.11 * np.exp(17.27*T_water_C/(237.3 + T_water_C))   # Saturated vapor pressure (hPa) 
q0 = 0.622 * e_sat /surface_pressure       # Specific humidity at saturation pressure (kg kg^-1)

ea = get_saturation_vapor_pressure(T_air_c)   # Saturation vapor pressure in hPa
ea = ea #* 100 

specific_humidity = const06*ea/(surface_pressure-0.377*ea)
print("specific_humidity = %2.2f kg/kg" % specific_humidity)

specific_humidity = 0.622 * vapor_pressure / surface_pressure  # Specific humidity of the air at height z_q above the water surface (kg kg^-1)
print("specific_humidity = %2.2f kg/kg" % specific_humidity)

humidity_at_saturation = 0.622 * e_sat / surface_pressure 

# Calculate density of the air 
Ra = gas_constant * (1 + 0.608*specific_humidity)


# rho_air = 100 * surface_pressure/(Ra*(T_air_c + 275.16))
# print("density of air (rho_air) = %2.2f kg/m^3" % rho_air)

# GOTM 
rho_air = 100 * surface_pressure/(gas_constant*celsius2kelvin(T_air_c)*(1.0+const06*specific_humidity))
print("density of air (rho_air) = %2.2f kg/m^3" % rho_air)

# Stability 
s0=0.25*(T_water_C - T_air_c)/(wind_speed+1.0e-10)**2
s=s0*abs(s0)/(abs(s0)+0.01)

if wind_speed < 2.2:
    ad = 0
    ah = 0 
    ae = 0
    bd = 1.08
    bh = 1.185
    be = 1.23
    ch = 0 
    ce = 0 
    pd = -0.15
    ph = -0.157
    pe = -0.16
elif wind_speed < 5: 
    ad = 0.771
    ah = 0.927
    ae = 9.69
    bd = 0.0858
    bh = 0.0546
    be = 0.0521
    ch = 0 
    ce = 0 
    pd = 1
    ph = 1
    pe = 1
elif wind_speed < 8:   
    ad = 0.867
    ah = 1.15
    ae = 1.18
    bd = 0.0667
    bh = 0.01
    be = 0.01
    ch = 0 
    ce = 0 
    pd = 1
    ph = 1
    pe = 1
elif wind_speed < 25:   
    ad = 1.2
    ah = 1.17
    ae = 1.196
    bd = 0.025
    bh = 0.0075
    be = 0.008
    ch = -0.00045
    ce = -0.0004
    pd = 1
    ph = 1
    pe = 1


eps = 1.0e-12

cdd=(ad + bd * np.exp(pd * np.log(wind_speed + eps)))*1e-3
chd=(ah + bh * np.exp(ph * np.log(wind_speed + eps)) + ch * (wind_speed-8.0)**2)*1.0e-3
ced=(ae + be * np.exp(pe * np.log(wind_speed + eps)) + ce * (wind_speed-8.0)**2)*1.0e-3

if s < 0:
    if s > -3.3:
        x = 0.1+0.03*s + 0.9*np.exp(4.8*s)
    else:
        x = 0 
    cdd = cdd * x
    chd = chd * x
    ced = ced * x
else:
    cdd=cdd*(1.0+0.47 * np.sqrt(s))
    chd=chd*(1.0+0.63 * np.sqrt(s))
    ced=ced*(1.0+0.63 * np.sqrt(s))

# Sensible heat flux 
qh = chd * SpecificHeatAir * rho_air * wind_speed *(T_water_C - T_air_c)      

# Latent heat flux
qe = ced * Lv * rho_air * wind_speed *(humidity_at_saturation - specific_humidity)

print("Sensible heat flux (qh) = %2.2f W/m^2" % qh)
print("Latent heat flux (qe) = %2.2f W/m^2" % qe) 

ustar = 0.02 m/s
zo = 0.00 m
Sensible heat flux (Qh) = 8.41 W/m^2
Latent heat flux (Qe) = 26.67 W/m^2
CEN = 0.001265
CHN = 0.001265
Sensible heat flux (qh) = -27.12 W/m^2
Latent heat flux (qe) = -89.48 W/m^2


specific_humidity = 0.01 kg/kg
specific_humidity = 0.01 kg/kg
density of air (rho_air) = 1.21 kg/m^3
Sensible heat flux (qh) = 34.36 W/m^2
Latent heat flux (qe) = 113.11 W/m^2


In [ ]:
# formulation from laekflux 

g = 9.81 # Acceleration due to gravity (m/s^2)
vonKarman = 0.41       # von Karman constant
Charnock = 0.013       # Charnock constant
gas_constant = 287.1   # gas constant for dry air J kg-1 K-1
nu_v  = 1.5e-5         # kinematic viscosity, m2 s-1


e_sat = 6.11 * np.exp(17.27*T_water_C/(237.3 + T_water_C))   # Saturated vapor pressure (hPa) 
q0 = 0.622 * e_sat /surface_pressure       # Specific humidity at saturation pressure (kg kg^-1)
# qz = 0.622*ez/surface_pressure             # Specific humidity of the air at height z_q above the water surface 
specific_humidity = 0.622 * vapor_pressure / surface_pressure  # Specific humidity of the air at height z_q above the water surface (kg kg^-1)

humidity_at_saturation = 0.622 * e_sat / surface_pressure  
# called qz 

# Gas constant for moist air 
Ra = gas_constant * (1 + 0.608*specific_humidity)
rho_air = 100 * surface_pressure/(Ra*(T_air_c + 275.16))

# Virtual temperature calculation
virtual_air_temp = celsius2kelvin(T_air_c) * (1 + 0.61 * vapor_pressure/sat_pressure)  # Virtual temperature (K)

# Density of the overlying air (Verburg and Antenucci, 2010)
rho_air = 100 * surface_pressure/(Ra*(T_air_c + 275.16))

# Latent heat of vaporization (J kg^-1)
# latent_heat_vaporization = 2.501e6 - 2370*T_water_C      

# kinematic viscosity, m2 s-1
KinV = (1./rho_air)*(4.94e-8*T_air_c + 1.7184e-5)

# called Uz in script 
# wind_speed = 0.5 # Wind speed in m/s (example value, replace with actual data)
wind_speed = np.clip(wind_speed, 0.2, None)  # Ensure wind speed is not less than 0.2 m/s

# Estimate initial values for ustar
ustar = wind_speed * np.sqrt(0.00104+0.0015/(1+np.exp((-wind_speed+12.5)/1.56)))
z0 = (Charnock*ustar**2/g) + (0.11*KinV/ustar)
print("ustar = %2.2f m/s" % ustar)
print("zo = %2.2f m" % z0)

height_wind_measurement = 10 
# for i in range(0, nwind):
#     ustar[i] = vonKarman * wind_speed[i]/np.log(height_wind_measurement/z0)
#     dummy = z0 
#     z0[i] = (Charnock*ustar[i]**2/g) + (0.11*KinV[i]/ustar[i])

# Calculate neutral transfer coefficients
C_DN = (ustar**2)/(wind_speed**2)
re = ustar*z0/KinV
zot = z0 * np.exp(-2.67*(re)**(0.25) + 2.57)
zoq = zot
C_HN = vonKarman * np.sqrt(C_DN)/(np.log(height_wind_measurement/zot)) 
C_EN = C_HN

# Calculate neutral transfer coefficients at 10 m
C_D10N = (vonKarman/np.log(10./z0)) * (vonKarman/np.log(10/z0)) 
C_E10N = (vonKarman**2)/(np.log(10/z0)*np.log(10/zoq))
C_H10N = C_E10N 
    
# calculate neutral latent and sensible heat fluxes, W/m2
alhN = rho_air * latent_heat_vaporization * C_EN * wind_speed * (humidity_at_saturation - specific_humidity)
ashN = rho_air * SpecificHeatAir * C_HN * wind_speed * (T_water_C-T_air_c) 
print("Sensible heat flux (Qh) = %2.2f W/m^2" % ashN)
print("Latent heat flux (Qe) = %2.2f W/m^2" % alhN)


print("CEN = %2.6f" % C_EN)
print("CHN = %2.6f" % C_HN)

# Monin - Obukhov length calculation
term1 = -rho_air * virtual_air_temp * (ustar**3)
term2 = vonKarman * g * (ashN/SpecificHeatAir + 0.61*celsius2kelvin(T_air_c)*alhN/latent_heat_vaporization)
monin_obukhov_length = term1/term2 
    


In [ ]:



# zo_prev = z0*1.1
# for i = range(1,length(Uz)) 
#     while abs((zo(i) - zo_prev(i)))/abs(zo_prev(i)) > 0.00001
#         ustar(i) = const_vonKarman.*Uz(i)/(log(hu./zo(i)))
#         dummy = zo(i)
#         zo(i)=(const_Charnock.*ustar(i).^2./const_Gravity) + (0.11*KinV(i)./ustar(i))
#         zo_prev(i) = dummy
#     end
# end
# term1 = -rho_z * u_star_a**3  * virtual_air_temp 
# term2 = vonKarman * g * (Qh/Cpa + 0.61 * celsius2kelvin(T_air_c)*Qe/Lv)
# monin_obukhov_length = 

# es = 6.11 * np.exp(17.27*T_air_c /237.3 + T_air_c)  # Saturated vapor pressure at height z (hPa) # check parenthesis on denominator? 
# ez = RH * es/100 

zeta_thres = 15
zetam = -1.574
zetat = -0.465

zeta  = 10/monin_obukhov_length

print("Monin-Obukhov length = %2.2f m" % monin_obukhov_length)
print(zeta)

idx_m_vu = zeta < zetam            # very unstable conditions zetaM
idx_m_u = zeta < 0 & zeta >= zetam # unstable conditions zetaM
idx_t_vu = zeta < zetat            # very unstable conditions zetaT
idx_t_u = zeta < 0 & zeta >= zetat # unstable conditions zetaT
idx_s = zeta >= 0 & zeta <= 1      # stable conditions
idx_vs = zeta > 1                  # very stable conditions


for i in range(0,20):
    z0 = (0.013 * ustar**2/g) + (0.11*KinV/ustar)
    re = ustar*z0/KinV
    xq = 2.67 * re**0.25 - 2.57
    xq = np.clip(xq, 0, None)  # Ensure xq is not negative

    zoq = z0/np.exp(xq)       
    zot = zoq 

    zeta  = 10/monin_obukhov_length
    zeta = np.clip(zeta, -zeta_thres, zeta_thres)  # Limit zeta to a maximum threshold

    if zeta< zetam: 
        term1 = wind_speed * vonKarman
        term2 = ((np.log((zetam*obu)/z0) - psi_zeng(1,zetam)) + ...
            1.14.*(((-zeta**0.333) - ((-zetam**0.333)))
        
        ustar = term1/term2 


In [ ]:
# Surface flux of momentum 
tau = C_dz * rho_z * u_z**2 

# Sensible heat
Qh = rho_z * C_pa * C_hz * u_z * (T_water_C - T_air_c)

# Latent heat
Qe = rho_z * L_v * C_ez * u_z * (q0 -qz)



In [ ]:




# # Cosine of solar zenith angle 
# sin1 = np.sin(latitude * 2*np.pi/360)
# sin2 = np.sin(sigma * 2*np.pi/360)
# cos1 = np.cos(latitude * 2*np.pi/360)
# cos2 = np.cos(sigma * 2*np.pi/360)
# cos3 = np.cos(H)
# cos_zenith = sin1*sin2 + cos1*cos2*cos3

   

x = 8.64153
inside 0.93387
zenith = 0.36572
Solar zenith angle (radians): 0.36571741461436347
Angle of refraction (radians): 0.27223701171508596
Albedo: 0.020279079871333296
